In [ ]:
# Install LIbraries
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [ ]:
# Imports
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
DPO patch applied.
GPU: Tesla T4


In [ ]:
# 5. Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"


def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [ ]:
# STAGE 1 DATA: PDF -> raw text dataset
# ============================================================

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages


def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by hyphen and newline
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove page-number-only lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Normalize spaces and paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)


def build_pdf_dataset(pdf_path: str) -> Dataset:
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])

        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()

            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)

In [ ]:
# 3. File path
non_instruction_data_path = "/content/non_instruction_data.pdf"
# instruction_data_path = "/content/instruction_dataset.jsonl"
# preference_data_path = "/content/preference_dataset.jsonl"

# for path in [non_instruction_data_path, instruction_data_path, preference_data_path]:
for path in [non_instruction_data_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# 4. Simple config

BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

MAX_SEQ_LENGTH = 512
SEED = 42
MIN_CHARS_PER_PARAGRAPH = 80

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
WARMUP_STEPS = 5
LOGGING_STEPS = 1

# Classroom demo steps. Increase for serious training.
STAGE1_MAX_STEPS = 30
STAGE2_MAX_STEPS = 30
STAGE3_MAX_STEPS = 30

STAGE1_LR = 2e-4
STAGE2_LR = 1e-4
STAGE3_LR = 5e-5

DPO_BETA = 0.1

OUTPUT_ROOT = "/content/unsloth_merge_reload_outputs"

STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_non_instruction_adapter"
STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_non_instruction_merged_model"

STAGE2_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage2_instruction_adapter"
STAGE2_MERGED_DIR  = f"{OUTPUT_ROOT}/stage2_instruction_merged_model"

STAGE3_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage3_dpo_adapter"
FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/stage3_dpo_final_merged_model"

for path in [
    OUTPUT_ROOT,
    STAGE1_ADAPTER_DIR,
    STAGE1_MERGED_DIR,
    STAGE2_ADAPTER_DIR,
    STAGE2_MERGED_DIR,
    STAGE3_ADAPTER_DIR,
    FINAL_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

In [ ]:
# ============================================================
# STAGE 1: Non-instruction continued pretraining
# ============================================================

print("\n==============================")
print("STAGE 1: PDF RAW TEXT TRAINING")
print("==============================")

stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

# Define the stage1_dataset by calling build_pdf_dataset
stage1_dataset = build_pdf_dataset(non_instruction_data_path)

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,

    seed=SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=stage1_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION PDF TRAINING")

save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE1_ADAPTER_DIR,
    merged_dir=STAGE1_MERGED_DIR,
    stage_name="Stage 1",
)

del stage1_trainer
del stage1_model
clear_gpu_memory()


STAGE 1: PDF RAW TEXT TRAINING
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338
PDF pages extracted: 4
Paragraph records: 4

Sample paragraph:
 Ticketing and SLA Management The IT Helpdesk serves as the single point of contact (SPOC) for all end-user technical inquiries, incidents, and service requests within the enterprise environment. All inbound communications, whether via the self-service portal, email, or telephone, must be logged immediately into the IT Service Management (ITSM) platform as a distinct and 

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Packing train dataset (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

...

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:37<00:00, 37.13s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_merge_reload_outputs/stage1_non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_merge_reload_outputs/stage1_non_instruction_merged_model


## Save Merged Model to Hugging Face Hub

To share your fine-tuned model, you can save it to the Hugging Face Model Hub. You'll need a Hugging Face token with write access. If you don't have one, create it on your Hugging Face profile settings under 'Access Tokens'.

Once you have your token, you can log in programmatically:

In [ ]:
# Log in to Hugging Face Hub
from huggingface_hub import login

# You will be prompted to enter your Hugging Face token
login()

Now, let's load the merged model from Stage 1 and push it to the Hugging Face Hub. Replace `'your_username/your_model_name'` with your desired repository name on Hugging Face.

In [ ]:
# Load the merged model from Stage 1
# We already loaded it in the previous cell as stage1_model_loaded and tokenizer_loaded
# model_to_push = stage1_model_loaded
# tokenizer_to_push = tokenizer_loaded

# Alternatively, if you're running this cell independently:
model_to_push, tokenizer_to_push = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Push the model and tokenizer to Hugging Face Hub
# IMPORTANT: Replace 'your_username/your_model_name' with your desired repository name
# The repository will be created if it doesn't exist.
# For example: 'your_username/TinyLlama_IT_Policy_Stage1_Merged'
repo_id = "mnayak1/non_instruction_merged_model"

print(f"Pushing model to Hugging Face Hub: {repo_id}")
model_to_push.push_to_hub(repo_id, tokenizer=tokenizer_to_push)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Pushing model to Hugging Face Hub: mnayak1/non_instruction_merged_model


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...aqet8ml/model.safetensors:   4%|4         | 31.8MB /  762MB            

Saved model to https://huggingface.co/mnayak1/non_instruction_merged_model


## Test Stage 1 Model

After non-instruction fine-tuning, the model has learned to better predict the next token based on the provided text. We can test its ability to complete a given text using the `generate_answer` helper function.

In [ ]:
# Load the merged model from Stage 1 for testing
# Note: stage1_model and tokenizer are deleted at the end of cell o1p1syPogsHP.
# If you're running this cell out of order, you might need to re-run the `load_unsloth_model_with_lora` step for the base model or the merged model.

print(f"Loading merged model from Stage 1: {STAGE1_MERGED_DIR}")
stage1_model_for_test, tokenizer_for_test = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Configure tokenizer padding if necessary
if tokenizer_for_test.pad_token is None:
    tokenizer_for_test.pad_token = tokenizer_for_test.eos_token
tokenizer_for_test.padding_side = "right"

print("Stage 1 merged model loaded successfully.")

instruction_for_test = "Continue the following text, focusing on IT policy details:"
input_text_for_test = "Users frequently encounter password expiration prompts due to the standard ninety-day security policy enforcement required by organizational compliance standards. When a user forgets their active directory password, they should first attempt to use the self-service password reset portal before contacting the helpdesk. The self-service portal requires the user to have previously registered their security questions or an alternate authentication method such as a mobile phone. If an account becomes locked after five consecutive failed login attempts, it will automatically unlock after thirty minutes without intervention. Helpdesk technicians can manually unlock accounts in the Active Directory."

print(f"\nInstruction: {instruction_for_test}")
print(f"Input: {input_text_for_test[:100]}...")

response = generate_answer(
    model=stage1_model_for_test,
    tokenizer=tokenizer_for_test,
    instruction=instruction_for_test,
    input_text=input_text_for_test,
    max_new_tokens=150
)

print(f"\nGenerated Response: {response}")

# Clear GPU memory after testing the Stage 1 model
del stage1_model_for_test
del tokenizer_for_test
clear_gpu_memory()

Loading merged model from Stage 1: /content/unsloth_merge_reload_outputs/stage1_non_instruction_merged_model
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Stage 1 merged model loaded successfully.

Instruction: Continue the following text, focusing on IT policy details:
Input: Users frequently encounter password expiration prompts due to the standard ninety-day security polic...

Generated Response: Technical support staff can remotely log into an account and unlock the account from a centralized administration console if a user forgets their account's password. This is a safer alternativ

### Verify Model on Hugging Face Hub

You can check if the model repository exists on Hugging Face by using the `huggingface_hub` library.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Assuming repo_id is still defined from the previous cell
# If not, you might need to redefine it: repo_id = "mnayak1/non_instruction_merged_model"

try:
    api.repo_info(repo_id=repo_id)
    print(f"Repository '{repo_id}' found on Hugging Face Hub.")
except Exception as e:
    print(f"Repository '{repo_id}' not found on Hugging Face Hub. Error: {e}")

Repository 'mnayak1/non_instruction_merged_model' found on Hugging Face Hub.
